In [ ]:
# Python example
import os
import pandas as pd
import pandasql

In [ ]:
# Upload data and categories
data_path = "../data/amazon-purchases.csv"
categories_path = "../data/categories.xlsx"
data = pd.read_csv(data_path)
categories = pd.read_excel(categories_path)

In [ ]:
# Create a sample of the data for display
# data_sample = data.sample(1000, random_state=42)
# data_sample.to_csv("../data/data_sample.csv", index=False)

In [ ]:
# Discovered formatting differences in the Category column. Some categories are uppercase while others are not. 
# This can cause issues when merging with the categories dataframe, which has all categories in uppercase.
not_uppercase_categories = data[data['Category'].str.isupper() == False]
not_uppercase_categories['Category'].unique()

In [ ]:
#Add categories to the data folder
#categories.to_csv("../data/categories.csv", index=True)

In [ ]:
departments = categories[['Department']].drop_duplicates().reset_index(drop=True)
departments['id'] = range(1, len(departments) + 1)
departments = departments[['id', 'Department']]
departments.columns = ['id', 'department']
departments.head()

In [ ]:
departments['id'].max()

In [ ]:
# Query the number of transactions per department and sort greatest to least
query = """
SELECT d.department, COUNT(*) as transaction_count
FROM data AS da
JOIN categories AS c ON da.Category = c.Category
JOIN departments AS d ON c.Department = d.department
GROUP BY d.department
ORDER BY transaction_count DESC
"""

result = pandasql.sqldf(query, locals())
result

,department,transaction_count
0,Food and Drink,219116
1,Apparel,190682
2,Electronics,189847
3,Personal Care,168089
4,Health,99291
5,Books,92658
6,Pet Supplies,86561
7,Home,84999
8,Kitchen,84727
9,Toys and Games,76894


In [ ]:
# Query the number of transactions per department for a specific month and sort greatest to least
query_august = """
SELECT d.department, COUNT(*) as transaction_count
FROM data AS da
JOIN categories AS c ON da.Category = c.Category
JOIN departments AS d ON c.Department = d.department
WHERE strftime('%m', da.[Order Date]) = '08'
GROUP BY d.department
ORDER BY transaction_count DESC
"""

result_august = pandasql.sqldf(query_august, locals())
result_august

In [ ]:
# This is the number of unique products in the dataset.
data['Title'].nunique()

824722

In [ ]:
# Upload departments to data folder
#  departments.to_csv("../data/departments.csv", index=False)

In [ ]:
### The following section explores creating sub department groups of product titles using 
#Natural Language Processing (NLP) techniques. This is an optional step that can be used 
#to further categorize products within the Exercise department.

from sentence_transformers import SentenceTransformer
exercise_categories = categories.loc[categories['Department'].eq('Exercise'), 'Category']
sentences = (
    data.loc[data['Category'].isin(exercise_categories), 'Title']
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(sentences)
print(embeddings)

In [ ]:
from sklearn.manifold import TSNE

reducer = TSNE(n_components=2, random_state=42)
embeddings_2d = reducer.fit_transform(embeddings)
import matplotlib.pyplot as plt

# sentences = ["This is an example sentence", "Each sentence is converted"]

# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
# embeddings = model.encode(sentences)

# Plot
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1])
plt.show()

In [ ]:
import plotly.express as px
import random

plot_df = pd.DataFrame({
    "x": embeddings_2d[:, 0],
    "y": embeddings_2d[:, 1],
    "title": sentences
})

sample_idx = random.sample(range(len(sentences)), min(30, len(sentences)))
plot_df["label"] = ""
plot_df.loc[sample_idx, "label"] = plot_df.loc[sample_idx, "title"].str[:40]

fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    text="label",
    hover_data={"title": True, "x": False, "y": False},
    opacity=0.6,
    width=1200,
    height=800
)

fig.update_traces(marker=dict(size=6), textposition="top center", textfont_size=9)
fig.show()

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

# Choose number of clusters - experiment with this
n_clusters = 15

kmeans = KMeans(n_clusters=n_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(embeddings)  # cluster in 384D, not 2D

# Plot points colored by cluster
fig, ax = plt.subplots(figsize=(14, 10))
scatter = ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                     c=cluster_labels, cmap='tab20', alpha=0.6, s=20)

# Label each cluster with a few representative titles
for cluster_id in range(n_clusters):
    idx = np.where(cluster_labels == cluster_id)[0]
    # Pick a few titles from this cluster to summarize it
    sample_titles = [sentences[i][:40] for i in idx[:3]]
    label = "\n".join(sample_titles)
    
    # Place label at centroid of cluster in 2D
    cx, cy = embeddings_2d[idx, 0].mean(), embeddings_2d[idx, 1].mean()
    ax.annotate(label, (cx, cy), fontsize=6, ha='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()